In [17]:
# BigQuery Data Cleaning Pipeline
# Imports ve Config
# Gerekli paketler: pip install google-cloud-bigquery pandas numpy pandas-gbq pyarrow db-dtypes
from google.cloud import bigquery
import pandas as pd
import numpy as np
import pandas_gbq
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

# Config
PROJECT_ID = "datawarehouseanalytics-480618"
DATASET_ID = "datawarehouseset"
OUTPUT_DIR = Path("cleaned_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Proje: {PROJECT_ID}")
print(f"Dataset: {DATASET_ID}")
print(f"Çıktı klasörü: {OUTPUT_DIR.absolute()}")

Proje: datawarehouseanalytics-480618
Dataset: datawarehouseset
Çıktı klasörü: c:\Users\acer\Desktop\Ptdeneme\cleaned_data


In [18]:
# BigQuery Client ve Tablo Listesi
client = bigquery.Client(project=PROJECT_ID)
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
tables_list = client.list_tables(dataset_ref)
# Temizlenmiş tabloları (cleaned_*) atla - BigQuery'da zaten mevcut, tekrar işleme
table_names = [table.table_id for table in tables_list if not table.table_id.startswith('cleaned_')]
print(f"İşlenecek tablolar: {table_names}")


İşlenecek tablolar: ['gold_dim_customer_segments', 'gold_dim_customers', 'gold_dim_products', 'gold_fact_sales']


In [19]:
df_sales = table_names[0]

In [20]:
# Cleaning Fonksiyonları
def standardize_columns(df):
    """Sütun isimlerini standartlaştır: lowercase, boşluk -> _, özel karakter temizle"""
    new_columns = {}
    seen = {}
    for col in df.columns:
        new_name = str(col).lower().strip()
        new_name = re.sub(r'\s+', '_', new_name)
        new_name = re.sub(r'[^a-z0-9_]', '', new_name)
        if not new_name:
            new_name = f"col_{list(df.columns).index(col)}"
        if new_name in seen:
            seen[new_name] += 1
            new_name = f"{new_name}_{seen[new_name]}"
        else:
            seen[new_name] = 0
        new_columns[col] = new_name
    return df.rename(columns=new_columns)

def trim_whitespace(df):
    """Object/string sütunlarda baştaki ve sondaki boşlukları temizle"""
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    return df

def remove_duplicates(df):
    """Tekrarlayan satırları kaldır"""
    return df.drop_duplicates()

def fix_dtypes(df):
    """Veri tiplerini düzelt: dbdate -> datetime64, sayısal sütunlar"""
    for col in df.columns:
        dtype_str = str(df[col].dtype)
        if 'dbdate' in dtype_str or 'date' in dtype_str.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif df[col].dtype == 'object':
            try:
                numeric = pd.to_numeric(df[col], errors='coerce')
                if numeric.notna().sum() > len(df) * 0.5:
                    df[col] = numeric
            except:
                pass
    return df

def handle_missing(df):
    """Eksik değerleri işle: %50+ eksik sütun drop, sayısal median, kategorik mode, boolean mode"""
    threshold = 0.5
    cols_to_drop = [col for col in df.columns if df[col].isna().mean() >= threshold]
    df = df.drop(columns=cols_to_drop, errors='ignore')
    for col in df.columns:
        if df[col].isna().any():
            if pd.api.types.is_bool_dtype(df[col]):
                mode_val = df[col].mode()
                df[col] = df[col].fillna(mode_val[0] if len(mode_val) > 0 else False)
            elif pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
            else:
                mode_val = df[col].mode()
                df[col] = df[col].fillna(mode_val[0] if len(mode_val) > 0 else 'Unknown')
    return df

def flag_outliers(df):
    """IQR yöntemi ile aykırı değerleri tespit et, outlier_flag sütunu ekle"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        df['outlier_flag'] = False
        return df
    outlier_mask = pd.Series([False] * len(df), index=df.index)
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outlier_mask |= (df[col] < lower) | (df[col] > upper)
    df['outlier_flag'] = outlier_mask
    return df

In [21]:
# Ana Pipeline Fonksiyonu
def clean_table(df, table_name):
    """Tüm cleaning adımlarını sırayla uygula"""
    df = df.copy()
    df = standardize_columns(df)
    df = trim_whitespace(df)
    df = remove_duplicates(df)
    df = fix_dtypes(df)
    df = handle_missing(df)
    df = flag_outliers(df)
    return df

In [22]:
# Döngü: Her tablo için yükle -> temizle -> CSV kaydet -> BigQuery'a yaz (opsiyonel)
report = []
for table_name in table_names:
    try:
        full_table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
        query = f"SELECT * FROM `{full_table_id}`"
        df_raw = client.query(query).to_dataframe()
        rows_before = len(df_raw)
        
        df_clean = clean_table(df_raw, table_name)
        rows_after = len(df_clean)
        duplicates_removed = rows_before - rows_after
        
        cleaned_table_name = f"cleaned_{table_name}"
        
        # CSV her zaman kaydedilir (billing olmadan da çalışır)
        csv_path = OUTPUT_DIR / f"{cleaned_table_name}.csv"
        df_clean.to_csv(csv_path, index=False)
        
        # BigQuery'a yaz - billing gerekir, hata verirse devam et
        bq_status = 'OK'
        try:
            destination = f"{DATASET_ID}.{cleaned_table_name}"
            pandas_gbq.to_gbq(df_clean, destination, project_id=PROJECT_ID, if_exists='replace')
        except Exception as bq_err:
            bq_status = f"BigQuery atlandı: {str(bq_err)[:80]}..."
            print(f"  ⚠ BigQuery yazılamadı (billing gerekebilir), CSV kaydedildi")
        
        report.append({
            'table': table_name,
            'rows_before': rows_before,
            'rows_after': rows_after,
            'duplicates_removed': duplicates_removed,
            'status': bq_status
        })
        print(f"✓ {table_name}: {rows_before} -> {rows_after} satır, {duplicates_removed} tekrar kaldırıldı, CSV: {csv_path.name}")
    except Exception as e:
        report.append({
            'table': table_name,
            'rows_before': None,
            'rows_after': None,
            'duplicates_removed': None,
            'status': str(e)
        })
        print(f"✗ {table_name}: HATA - {e}")

  ⚠ BigQuery yazılamadı (billing gerekebilir), CSV kaydedildi
✓ gold_dim_customer_segments: 18484 -> 18484 satır, 0 tekrar kaldırıldı, CSV: cleaned_gold_dim_customer_segments.csv
  ⚠ BigQuery yazılamadı (billing gerekebilir), CSV kaydedildi
✓ gold_dim_customers: 18484 -> 18484 satır, 0 tekrar kaldırıldı, CSV: cleaned_gold_dim_customers.csv
  ⚠ BigQuery yazılamadı (billing gerekebilir), CSV kaydedildi
✓ gold_dim_products: 295 -> 295 satır, 0 tekrar kaldırıldı, CSV: cleaned_gold_dim_products.csv
  ⚠ BigQuery yazılamadı (billing gerekebilir), CSV kaydedildi
✓ gold_fact_sales: 60398 -> 60398 satır, 0 tekrar kaldırıldı, CSV: cleaned_gold_fact_sales.csv


In [ ]:
# Özet Rapor
report_df = pd.DataFrame(report)
print("\n=== Temizleme Özet Raporu ===")
print(report_df.to_string(index=False))
print(f"\nYerel dosyalar: {OUTPUT_DIR.absolute()}")